# Currency Analyst Agent

This notebook walks through the **Currency Analyst Agent** as structured in this repository — a CrewAI-powered financial intelligence system that answers natural-language questions about **real-time foreign exchange**.

You ask something like *“What is the current exchange rate between USA and Germany?”* and the system:

1. Routes the question through a **FastAPI** backend  
2. Runs a **CrewAI** agent with specialized FX tools  
3. Fetches live data from **ExchangeRate-API**  
4. Returns a clear, structured answer (optionally via the **Streamlit** chat UI)

Live Spot rates come from tools — not from the model’s memory.

## 1. Environment setup

Install the same stack used by the project (`crewai`, FastAPI, Streamlit, etc.), load API keys from `.env`, and put the project root on `sys.path` so package imports resolve the way they do after `uv sync`.

Required secrets:

- `EXCHANGE_RATE_API_KEY` — ExchangeRate-API  
- `OPENAI_API_KEY` — LLM used by the CrewAI agent

In [9]:
# uv venvs do not include pip by default, so %pip / python -m pip fails with
# "No module named pip". Install into THIS kernel's venv with uv instead:
import sys
!uv pip install -q --python {sys.executable} "crewai[tools]" fastapi streamlit python-dotenv requests pydantic uvicorn

In [10]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "experimentation_notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
for path in (PROJECT_ROOT, SRC_PATH):
    path_str = str(path)
    
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

load_dotenv(PROJECT_ROOT / ".env")
load_dotenv()  # also pick up a parent .env if present

os.makedirs(PROJECT_ROOT / "output", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXCHANGE_RATE_API_KEY set:", bool(os.getenv("EXCHANGE_RATE_API_KEY")))
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

PROJECT_ROOT: /Users/user/Developer/genai_works/src/currency_analyst_agent
EXCHANGE_RATE_API_KEY set: True
OPENAI_API_KEY set: True


## 2. Tool input schemas

Pydantic models define what each CrewAI tool accepts. This mirrors `src/currency_analyst_crew/tools/tool_schema.py`.

- **SupportedCurrenciesInput** — no fields (list-all endpoint needs no arguments)  
- **CurrencyConverterInput** — `from_currency` and `to_currency` ISO codes

In [11]:
from pydantic import BaseModel, Field


class SupportedCurrenciesInput(BaseModel):
    """Input schema for SupportedCurrenciesTool. No inputs required."""
    pass  # no user input needed for this tool.


class CurrencyConverterInput(BaseModel):
    """Input schema for CurrencyConverterTool."""

    from_currency: str = Field(
        ...,
        description="The base or source currency code (e.g., USD, NGN) to convert from.",
    )
    to_currency: str = Field(
        ...,
        description="The target currency code (e.g., EUR) to convert to.",
    )


CurrencyConverterInput(from_currency="USD", to_currency="EUR")

CurrencyConverterInput(from_currency='USD', to_currency='EUR')

## 3. Custom tools (ExchangeRate-API)

These tools ground the agent in live market data. Code mirrors `src/currency_analyst_crew/tools/custom_tool.py`.

| Tool | Endpoint | Returns |
| --- | --- | --- |
| `SupportedCurrenciesTool` | `/v6/{key}/codes` | `CODE - Name` lines |
| `CurrencyConverterTool` | `/v6/{key}/pair/{from}/{to}` | Spot rate (`1 FROM = rate TO`) |

In [12]:
import requests
from typing import Type
from crewai.tools import BaseTool


exchange_rate_api_key = os.getenv("EXCHANGE_RATE_API_KEY")


class SupportedCurrenciesTool(BaseTool):
    """
    Tool for retrieving all supported currency codes and their corresponding
    countries from the Exchange Rate API. Exchange Rate API supports 166 currency codes.

    This tool fetches the complete list of currency codes and corresponding currency name
    that the API can handle for real-time conversion. It is designed to provide
    reference context for the LLM, ensuring it only works with valid currencies
    when performing conversions or comparisons. This tool does not provide
    exchange rates or historical data.
    """

    name: str = "Supported Currencies Tool"
    description: str = (
        "Fetches all currency codes and their corresponding currency name supported "
        "by the Exchange Rate API. Useful for providing the LLM with valid currency "
        "context. Does not require any input and does not return exchange rates."
    )
    args_schema: Type[BaseModel] = SupportedCurrenciesInput
    api_key: str = exchange_rate_api_key

    def _run(self) -> str:
        url = f"https://v6.exchangerate-api.com/v6/{self.api_key}/codes"
        response = requests.get(url)

        if response.status_code != 200:
            return "Failed to fetch supported currency codes."

        data = response.json()
        supported_codes = data.get("supported_codes", [])
        output_lines = [f"{code} - {country}" for code, country in supported_codes]
        return "\n".join(output_lines)


class CurrencyConverterTool(BaseTool):
    """
    Tool for retrieving the real-time exchange rate between two supported
    currencies.

    This tool fetches the latest available exchange rate for a specified
    currency pair and returns the current rate. It is designed for real-time
    currency rate lookup and amount-based conversions via amount × rate
    (where the returned rate is the converted value of 1 unit of the base
    currency). It does not support historical data, trend analysis, or
    future predictions.
    """

    name: str = "Currency Converter Tool"
    description: str = (
        "Retrieves the current real-time exchange rate between a specified "
        "base or source currency and a target currency. Returns the latest "
        "rate for the currency pair, which can be used for amount-based "
        "conversions as amount × rate (rate equals the value of 1 unit of "
        "the base currency). Does not provide historical or predictive data."
    )
    args_schema: Type[BaseModel] = CurrencyConverterInput
    api_key: str = exchange_rate_api_key

    def _run(self, from_currency: str, to_currency: str) -> str:
        url = (
            f"https://v6.exchangerate-api.com/v6/{self.api_key}"
            f"/pair/{from_currency}/{to_currency}"
        )
        response = requests.get(url)

        if response.status_code != 200:
            return "Failed to fetch current exchange rates."

        data = response.json()
        conversion_rate = data.get("conversion_rate")
        target_code = data.get("target_code")

        if conversion_rate is None or target_code != to_currency:
            return "Invalid currency code."

        return (
            f"Current exchange rate: 1 {from_currency} = {conversion_rate} {to_currency}"
        )


supported_currencies_tool = SupportedCurrenciesTool()
currency_converter_tool = CurrencyConverterTool()
print("Tools ready:", supported_currencies_tool.name, "|", currency_converter_tool.name)

Tools ready: Supported Currencies Tool | Currency Converter Tool


### Smoke-test the tools (optional)

Call the tools directly, no LLM is required. Useful to verify your ExchangeRate-API key before running the full crew.

In [13]:
# Preview a few supported currencies
codes_preview = supported_currencies_tool._run().splitlines()[:8]
print("Supported currencies (sample):")
print("\n".join(codes_preview))

print()
print(currency_converter_tool._run("USD", "EUR"))

Supported currencies (sample):
AED - UAE Dirham
AFN - Afghan Afghani
ALL - Albanian Lek
AMD - Armenian Dram
ANG - Netherlands Antillian Guilder
AOA - Angolan Kwanza
ARS - Argentine Peso
AUD - Australian Dollar

Current exchange rate: 1 USD = 0.866 EUR


## 4. Agent & task configuration (YAML)

Define the agent **role / goal / backstory / LLM** and the task specs **inline in this notebook** (same content as the repo YAML).

The next cell:
1. Keeps the YAML as editable Python strings  
2. Writes them to `experimentation_notebook/config/` so `@CrewBase` can load them from the notebook CWD  

Shared input placeholder: **`{query}`**.

In [19]:
from pathlib import Path

# agents.yaml (role, goal, backstory, llm)
AGENTS_YAML = """
currency_analyst:
  role: >
    Real-Time Currency Analyst for {query}
  goal: >
    Provide accurate and insightful information about current exchange rates
    and relationships between currencies in real time on {query}.
    If the user request have been answered using any of the tools, and you are confident in your answer or response,
    respond quickly and concisely without invoking any more tools.
  backstory: >
    You are a financial intelligence AI specialized in analyzing currencies.
    You excel at fetching real-time exchange rate data and interpreting
    currency relationships at the current moment on {query}. You provide clear and
    concise insights for users seeking to understand the relative strength
    of currencies across various countries or perform instant conversions.
  llm: openai/gpt-4o-mini
"""

# tasks.yaml
TASKS_YAML = """
supported_currencies_task:
  description: >
    Retrieve and present a list of all supported currencies available through
    the currency exchange API (exchangerate api). Focus on:
    1. Listing all the currency codes (e.g., USD, EUR, NGN) supported by exchangerate api.
    2. Including the corresponding country or region for each currency.
    3. Ensuring the list reflects only currencies that can be used for
       real-time conversion or currencies provided by the exchangerate api within the system.

    This task is intended to provide reference context for users and the agent on {query},
    and should not include exchange rate values or historical information. You should only consider
    this tool or task if information about the supported currencies is clearly required to answer user queries.
  expected_output: >
    A structured markdown document that:
    - Lists all supported currency codes.
    - Maps each currency code to its country or region.
    - Is clearly organized for easy reference.
    - Can be used as contextual knowledge for validating user queries.
  agent: currency_analyst

real_time_currency_task:
  description: >
    Perform real-time currency analysis on {query} based on current exchange rates. Focus on:
    1. Providing accurate conversions between any two currencies upon request.
    2. Comparing the relative strength of currencies at the current moment.
    3. Explaining currency relationships in clear and concise language.
    4. Offering actionable insights for users seeking to understand the current
       status of currencies across different countries.

    Always use the latest available exchange rate data to provide response on {query}. Do not use historical
    or predictive data beyond the current market snapshot.
  expected_output: >
    A text-based response on {query} that:
    - Converts currencies accurately as requested by the user.
    - Compares and explains the relative strength of currencies in real-time.
    - Provides clear, concise, and structured insights about currency relationships.
    - Uses markdown formatting where appropriate for clarity (headings, lists, bold).
  agent: currency_analyst
"""



print("\n=== agents.yaml ===\n")
print(AGENTS_YAML.strip())
print("\n=== tasks.yaml ===\n")
print(TASKS_YAML.strip())


=== agents.yaml ===

currency_analyst:
  role: >
    Real-Time Currency Analyst for {query}
  goal: >
    Provide accurate and insightful information about current exchange rates
    and relationships between currencies in real time on {query}.
    If the user request have been answered using any of the tools, and you are confident in your answer or response,
    respond quickly and concisely without invoking any more tools.
  backstory: >
    You are a financial intelligence AI specialized in analyzing currencies.
    You excel at fetching real-time exchange rate data and interpreting
    currency relationships at the current moment on {query}. You provide clear and
    concise insights for users seeking to understand the relative strength
    of currencies across various countries or perform instant conversions.
  llm: openai/gpt-4o-mini

=== tasks.yaml ===

supported_currencies_task:
  description: >
    Retrieve and present a list of all supported currencies available through
    

## 5. The crew: agent, tasks, sequential process

Define the crew **in this notebook** (same structure as `src/currency_analyst_crew/crew.py`).

- One agent: **`currency_analyst`** with both FX tools  
- Two sequential tasks: supported currencies → real-time analysis  
- Real-time task writes markdown to `output/report.md`

`@CrewBase` will load the YAML you wrote in the previous cell from `experimentation_notebook/config/`.

In [20]:
from typing import List
from crewai import Agent, Task, Crew, Process
from crewai.project import CrewBase, agent, crew, task
from crewai.agents.agent_builder.base_agent import BaseAgent

# Reuse tools from earlier cells (or rebuild if needed)
supported_currencies_tool = SupportedCurrenciesTool()
currency_converter_tool = CurrencyConverterTool()


@CrewBase
class CurrencyAnalystCrew:
    """Currency Analyst Crew for real-time exchange rate analysis and insights"""

    agents: List[BaseAgent]
    tasks: List[Task]

    @agent
    def currency_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config["currency_analyst"],  # type: ignore[index]
            verbose=True,
            memory=True,
            tools=[
                supported_currencies_tool,
                currency_converter_tool,
            ],
        )

    @task
    def supported_currencies_task(self) -> Task:
        return Task(
            config=self.tasks_config["supported_currencies_task"],  # type: ignore[index]
        )

    @task
    def real_time_currency_task(self) -> Task:
        return Task(
            config=self.tasks_config["real_time_currency_task"],  # type: ignore[index]
            markdown=True,
            output_file="output/report.md",
        )

    @crew
    def crew(self) -> Crew:
        """Creates the Currency Analyst crew"""
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
        )


print("CurrencyAnalystCrew defined (loads notebook config/*.yaml).")
print("Agent keys:", list(CurrencyAnalystCrew().agents_config.keys()))
print("Task keys:", list(CurrencyAnalystCrew().tasks_config.keys()))

CurrencyAnalystCrew defined (loads notebook config/*.yaml).
Agent keys: ['currency_analyst']
Task keys: ['supported_currencies_task', 'real_time_currency_task']


## 6. Crew entrypoint: `run(inputs: dict)`

This mirrors `src/currency_analyst_crew/main.py`.

- Input contract: `{"query": "..."}`  
- Output: raw text from the crew (`result.raw`)  
- Same function the FastAPI route calls

In [21]:
import os

os.makedirs("output", exist_ok=True)


def run(inputs: dict):
    """
    Run the currency analyst crew defined in this notebook.
    """
    result = CurrencyAnalystCrew().crew().kickoff(inputs=inputs)
    return result.raw


print("run() ready. Expected input key: 'query'")

run() ready. Expected input key: 'query'


### Kick off the crew

Run a natural-language query through the full agent pipeline. This calls the LLM and ExchangeRate-API (requires both API keys).

In [22]:
inputs = {
    "query": (
        "what is the current exchange rate between USA currency and Germany currency? "
        "Also, provide insights on factors that might have influenced this rate recently."
    )
}

# Alternative:
# inputs = {"query": "i would like to know all the supported currency codes."}

analysis = run(inputs)
print(analysis)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: CurrencyAnalystCrew                                                                                      │
│  ID: 85546aad-ee56-4051-a616-806cc0814275                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: supported_currencies_task                                                                                │
│  ID: 008744d9-77c9-4dff-b027-6fce6c9587aa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-YPRYT***************************************g3sA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-YPRYT***************************************g3sA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided:            │
│  sk-YPRYT***************************************g3sA. You can find your API key at                              │
│  https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key',    │
│  'param': None}, 'status': 401}                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: supported_currencies_task                                                                                │
│  Agent: Real-Time Currency Analyst for what is the current exchange rate between USA currency and Germany       │
│  currency? Also, provide insights on factors that might have influenced this rate recently.                     │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: CurrencyAnalystCrew                                                                                      │
│  ID: 85546aad-ee56-4051-a616-806cc0814275                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.